<img src="images\Logo_UCLL_ENG_RGB.png" style="background-color:white;" />

# Data Analytics & Machine learning

Lecturers: Aimée Lynn Backiel, Lies Bollens, Fien Van de Woestijne

Academic year 2026-2027

## Lab 7 Part 1: Machine learning, part 3

### Lecture outline

1. Recap of previous weeks
2. Automating machine learning pipelines with sci-kit learn
3. Model evaluation 
4. Switching from regression to classification

The focus of last lab was on using sci-kit learn's tools to made models. On top of that we saw different important performance metrics for regression and how to use them in sci-kit learn. Our overarching goal remains the same: we want to try out various approaches and select the best one afterwards. This lecture we will continue that.

### Recap of last lecture(s)

#### Lab 1

1. We ensured we had a valid Python installation.
2. We learnt what a virtual environment is:
   * Isolated Python executable and packages.
   * We created a virtual environment.
3. Absolute path vs relative path recap.
4. Recap of data structures in Python
5. Installed Pandas
6. Learnt how to read data
7. Learnt how to calculate mean, mode, median etc.
8. Basic exploration of the 4 variables

#### Lab 2
1. Wrapped up computing summary statistics (mean, median, mode, ...)
2. Learnt how to deal with outliers 
3. Focused on exploration of data

#### Lab 3
1. Univariate data visualization using Matplotlib
   1. Figures and axes
   2. Histograms
   3. Box plots
   4. Bar charts
2. Multivariate data visualization using Seaborn
   1. Scatter plots
   2. Small multiples
   3. Color coding

#### Lab 4
1. Intro to machine learning using scikit-learn
   1. Preprocessing
      1. One Hot encoding
      2. Scaling
      3. Outliers
   2. Regression

#### Lab 5
1. Preprocessing with scikit-learn
   1. ColumnTransformer: Apply a transformation to specific columns.
   2. Pipeline: Do several transformations after each other
2. Evaluation:
   1. Why the mean of the error is a bad idea
   2. Mean absolute error
   3. Mean squared error

### The case

Ada Turing Travelogue, or as everyone calls her, Ada just started working part time at her parents travel agency. She has a keen understanding and interest of everything related to applied computer science ranging from server & system management to full stack software development. Through database foundations she already understands how to query data and programming 1 and 2 covered the essentials about the Python programming language. Recently she has just decided to start learning about data analytics & machine learning as well.

She uses her skills to connect to the travel agency's database where she finds many, normalized, tables. Ada recalls what she learnt in database foundations and performs all the correct joins. Afterwards she saves the data in the `data/` folder.


She finds the following dataset:

| Column Name          | Description                                                                                       |
| -------------------- | ------------------------------------------------------------------------------------------------- |
| SalesID              | Unique identifier for each sale.                                                                  |
| Age                  | Age of the traveler.                                                                              |
| Country              | Country of origin of the traveler.                                                                |
| Membership_Status    | Membership level of the traveler in the booking system; could be 'standard', 'silver', or 'gold'. |
| Previous_Purchases   | Number of previous bookings made by the traveler.                                                 |
| Destination          | Travel destination chosen by the traveler.                                                        |
| Stay_length          | Duration of stay at the destination.                                                              |
| Guests               | Number of guests traveling (including the primary traveler).                                             |
| Travel_month         | Month in which the travel is scheduled.                                                           |
| Months_before_travel | Number of months prior to travel that the booking was made.                                       |
| Earlybird_discount   | Boolean flag indicating whether the traveler received an early bird discount.                     |
| Package_Type         | Type of travel package chosen by the traveler.                                                    |
| Cost                 | Calculated cost of the travel package.                                                            |
| Margin | The cost (for the traveler) - what the travel agency pays. |
 | Additional_Services_Cost| The amount of additional services (towels, car rentals, room service, ...) that was bought during the trip. |


#### Our challenge

Before getting into harder use cases we will start off by predicting the cost of a given stay. Right now Ada's parents do this manually automating this task would already be a big help to their business.

<center>
<img src="https://www.datascience-pm.com/wp-content/uploads/2021/02/CRISP-DM.png" style="background-color:white;width:50%">
</center>

### Machine learning with sci-kit learn


<center>
<img src="https://scikit-learn.org/stable/_images/grid_search_workflow.png" style="background-color:white;width:50%">
</center>

##### ❓ What have we done so far of the image below? What stages have we completed?


We hebben de dataset opgesplitst in een train- en testset (`train_test_split`), we hebben modellen getraind op de trainingsdata (met preprocessing in een pipeline) en we hebben ze geëvalueerd op de testset.

Wat nog ontbreekt is het middelste deel van de figuur: **cross validation** op de trainingsset om parameters/modellen te kiezen. Tot nu toe kozen we het "beste" model door rechtstreeks naar de testset te kijken.

We will continue from last lecture. We covered two cornerstones of sklearn namely:

1. `make_column_transformer`, this allows you to specify preprocessing you want to apply for different columns. E.g., scaling for numeric columns and one hot encoding for categorical.
2. `make_pipeline`, this allows you to compose several steps. So our first step could be preprocessing and the second our ML model. A pipeline is an end-to-end object that lets you go from raw data to a model.

Summary of last lab's code:

In [1]:
import pandas as pd # by convention
pd.options.display.float_format = '{:.2f}'.format
from sklearn.model_selection import train_test_split
import plotly.express as px
import numpy  as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [2]:
travel_dataset = pd.read_csv("data/lab_6_dataset.csv")
X = travel_dataset.drop(columns="cost") 
y = travel_dataset["cost"] 
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [3]:
cat_columns = ["package_Type", "destination", "country"]
numeric_columns = ["guests", "age", "stay_length"]

preprocessing = make_column_transformer(
    (StandardScaler(), numeric_columns),
    (OneHotEncoder(sparse_output=False), cat_columns),
    remainder="drop"
)

In [4]:
lin_reg_pipe = make_pipeline(preprocessing, LinearRegression())

In [5]:
lin_reg_pipe.fit(X_train, y_train)
predictions_lin_reg_train = lin_reg_pipe.predict(X_train)
predictions_lin_reg_test = lin_reg_pipe.predict(X_test)

#### Model evaluation

_The previous lab session covered evaluation, but this chapter is repeated here.
For those that have finished the final evaluation section of the previous lab, you can skip ahead to "Finding the best model: feature engineering"_.

Up until now we've only *qualitatively* judged the quality of our models by looking at our predicted versus actual plot. We're interested in having a single number that summarizes the the performance of our model. The reason is that investigating graphs doesn't scale well.

##### Attempt 0: taking the mean of the error

Our first intuition might be to take the difference of the predictions and the actual values. This is called the **error** or the **residual**. After we have this value we may be tempted to take the mean. 

In [6]:
error = predictions_lin_reg_test - y_test
error

33553    390.40
9427     532.95
199     -400.79
12447    -41.46
39489    244.13
          ...  
28567   -335.01
25079   -506.66
18707   -195.06
15200    131.65
5857     342.58
Name: cost, Length: 10000, dtype: float64

In [7]:
np.mean(error)

np.float64(-4.869325013024942)

In [8]:
px.histogram(error, title="distribution of the errors of linear regression on the test set.")

##### ❓ Why is the mean of residuals misleading?


Positieve en negatieve fouten heffen elkaar op. Een model dat de helft van de tijd 1000 te hoog en de helft 1000 te laag voorspelt heeft een gemiddelde fout van 0, terwijl het eigenlijk heel slecht is. Hier is de gemiddelde fout ongeveer -5, wat doet uitschijnen dat het model bijna perfect is, terwijl de histogram toont dat de fouten tot honderden euro's gaan.

##### ❓ How can we better quantify errors?

We moeten ervoor zorgen dat alle fouten positief zijn voor we het gemiddelde nemen. Dat kan door de absolute waarde te nemen (MAE) of door te kwadrateren (MSE / RMSE).

##### Attempt 1: taking the mean of the absolute error (MAE)

In [9]:
np.mean(np.abs(error))

np.float64(302.04813893375064)

In [10]:
px.histogram(np.abs(error), title="distribution of the absolute errors of linear regression on the test set.")

This approach is more informative as it provides a clearer picture of how much error is present on average in our predictions.

By considering the MAE, we obtain a useful summary statistic for model performance that is easy to understand and directly interpretable in terms of the problem at hand.

In practice squaring the errors is more common.

##### Attempt 2: Taking the mean of the squared errors (MSE)

Squaring the errors, as shown in the equation: $(-2)² = 4 = (2)²$, ensures that all error values are positive.

This method has the main advantage that large errors are amplified more than smaller ones, which can be particularly important in cases where larger deviations are less tolerable.


The MSE can be calculated as follows:

In [11]:
np.mean(np.square(error))

np.float64(164891.60349428872)

Since MSE can result in large numbers that are difficult to interpret, we often take the square root to obtain the Root Mean Squared Error (RMSE), which has the same units as the original values:

*Note: the root is $\sqrt(x)$.*

In [12]:
rmse = np.sqrt(np.mean(np.square(error)))

In [13]:
px.histogram(np.square(error), title="distribution of the square errors of linear regression on the test set.")

##### ❓ What are downsides of the MSE? There are two, but one is harder to come up with.

1. **Moeilijk te interpreteren**: de eenheid is in het kwadraat (euro²), dus een MSE van ±165 000 zegt op zich niet veel. Daarom nemen we meestal de wortel (RMSE).
2. **Gevoelig voor uitschieters**: omdat fouten gekwadrateerd worden, wegen enkele hele grote fouten veel zwaarder door dan veel kleine fouten. Een paar uitschieters kunnen de MSE dus helemaal domineren, ook al doet het model het voor de rest goed.

##### Summary

Our advice:

* Compute both the MAE and the RMSE. 
* The one you should focus on the most is typically the RMSE. 
* If there are error outliers then the difference between the MAE and the RMSE is likely going to be large. In that case it is typically more interesting to look at the MAE. 
 
Typically, RMSE is preferred because it is more sensitive to large errors, which can be critical in applications where such errors are especially problematic, like in autonomous vehicle guidance systems. However, if your model is prone to outliers, or if large errors are less impactful, MAE can be a more relevant metric. Always consider the specific context of your application when choosing your primary evaluation metric.

##### model evaluation using sci-kit learn


sci-kit learn offers all of these functions out of the box. All you need to do is remember their name.

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

##### ❓ Use the pipeline approach discussed above and the mean_absolute_error and mean_squared_error for the following models: RandomForestRegressor, HistGradientBoostingRegressor, DecisionTreeRegressor and LinearRegression. You can build on the function that you created above that returned results for each of the regressors.

In [15]:
# Functies uit labo 5, licht aangepast: root_mean_squared_error bestaat nu ook in sklearn
def train_models(models: dict, preprocessing, X_train, y_train, X_test) -> list[tuple]:
    """Traint elk model in een pipeline en geeft (naam, train-voorspellingen, test-voorspellingen) terug."""
    results = []
    for name, model in models.items():
        pipe = make_pipeline(preprocessing, model)
        pipe.fit(X_train, y_train)
        predictions_train = pipe.predict(X_train)
        predictions_test = pipe.predict(X_test)
        results.append((name, predictions_train, predictions_test))
    return results


def evaluate_models(results: list[tuple], y_train, y_test) -> pd.DataFrame:
    """Berekent MAE en RMSE op train- en testset voor elk model in results."""
    rows = []
    for name, predictions_train, predictions_test in results:
        rows.append({
            "model": name,
            "MAE train": mean_absolute_error(y_train, predictions_train),
            "MAE test": mean_absolute_error(y_test, predictions_test),
            "RMSE train": root_mean_squared_error(y_train, predictions_train),
            "RMSE test": root_mean_squared_error(y_test, predictions_test),
        })
    return pd.DataFrame(rows).set_index("model")


models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

results = train_models(models, preprocessing, X_train, y_train, X_test)
evaluate_models(results, y_train, y_test)

,MAE train,MAE test,RMSE train,RMSE test
model,,,,
Linear Regression,306.98,302.05,415.34,406.07
Decision Tree,12.20,279.88,51.05,412.46
Random Forest,91.42,235.02,124.36,314.26
Gradient Boosting,215.79,222.29,270.61,283.30


##### ❓ Comment on the behavior of the models. Are they overfitting? Underfitting?

* **Linear regression**: train- en testfout liggen dicht bij elkaar (RMSE ±415 vs ±406), maar ze zijn allebei hoog. Het model is te simpel om de patronen te vatten → **underfitting**.
* **Decision tree**: bijna perfect op de trainingsset (MAE ±12) maar slechter dan lineaire regressie op de testset → zware **overfitting**, de boom heeft de trainingsdata van buiten geleerd.
* **Random forest**: ook een groot verschil tussen train (RMSE ±124) en test (±314), dus ook overfitting, maar minder erg dan één boom omdat er over veel bomen uitgemiddeld wordt. Op de testset wel al een stuk beter.
* **Gradient boosting**: train en test liggen dicht bij elkaar (±271 vs ±283) én het is de laagste testfout. Dit model heeft de beste balans tussen under- en overfitting.

#### Finding the best model: feature engineering

Recall in the previous session we made a number of observations:

* If you're visiting the same country as you're from the destination seemed to be cheaper
* If you're traveling in approximately the same continent it's also cheaper
* There might be the case between age and month.
* Maybe we should look at age in groups.

In [16]:
destination_to_country = {
        "New York": "USA",
        "Rome": "Italy",
        "Paris": "France",
        "Tokyo": "Japan",
        "Cairo": "Egypt",
        "Sydney": "Australia",
        "Rio": "Brazil",
        "Cape Town": "South Africa",
    }
country_to_continent = {
        "USA": "America",
        "UK": "EMEA",
        "France": "EMEA",
        "Canada": "America",
        "Australia": "Asia",
        "Germany": "EMEA",
        "Spain": "EMEA",
        "Italy": "EMEA",
    }
destination_to_continent = {
        "New York": "America",
        "Rome": "EMEA",
        "Paris": "EMEA",
        "Tokyo": "Asia",
        "Cairo": "EMEA",
        "Sydney": "Asia",
        "Rio": "America",
        "Cape Town": "Africa",
    }

##### ❓ Use Pandas to make a variables to indicate if they traveled within the country (traveled to the country that they are from), and a variable to indicate if they traveled within the continent (traveled to a country that is in the continent that their country of origin is in). 

##### HINT1: Look at [the map method for Pandas series](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html). 

##### HINT2: Remember, a series is simply a column so , `df[column]` gives you a series.

In [17]:
# Op een kopie werken zodat X_train / X_test (die al gesplitst zijn) niet veranderen
travel_with_matches = travel_dataset.copy()

destination_country = travel_with_matches["destination"].map(destination_to_country)
travel_with_matches["country_match"] = travel_with_matches["country"] == destination_country

origin_continent = travel_with_matches["country"].map(country_to_continent)
destination_continent = travel_with_matches["destination"].map(destination_to_continent)
travel_with_matches["continent_match"] = origin_continent == destination_continent

travel_with_matches[["country", "destination", "country_match", "continent_match"]].head(10)

,country,destination,country_match,continent_match
0,Italy,Tokyo,False,False
1,Canada,Cairo,False,False
2,Australia,Paris,False,False
3,Germany,Paris,False,True
4,UK,Rio,False,False
5,Canada,Cairo,False,False
6,Australia,Rome,False,False
7,Australia,Cape Town,False,False
8,UK,New York,False,False
9,UK,Paris,False,True


When trying to add these variables there is some friction. They don't fit into our sci-kit learn `Pipeline` workflow nicely. We could add them to our entire dataset before splitting. Adding variables to the entire dataset is not risk-free. Doing so may lead to the methodological error we spoke about previously **data leakage**. In the scope of this course it's fine to use this approach to *add* variables. We'll briefly show you the more principled way, but you don't need to know this for the exam. It involves creating a custom `Transformer` which we can then compose in our pipeline as usual.

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin
from dataclasses import dataclass

@dataclass
class CountryMapping(TransformerMixin, BaseEstimator):
    country_to_continent: dict[str, str]
    destination_to_country: dict[str, str]
    destination_to_continent: dict[str, str]

    def fit(self, X, y=None):
        return self
    
    def transform(self, X: pd.DataFrame):
        _X = X.copy() # so we don't change our input data frame
        _X['country_match'] = _X['country'] == _X['destination'].map(self.destination_to_country)
        _X['continent_match'] = _X['country'].map(self.country_to_continent) == _X['destination'].map(self.destination_to_continent)
        return _X

In [19]:
country_map = CountryMapping(country_to_continent, destination_to_country, destination_to_continent)

In [20]:
country_map.fit_transform(X_train)

,sales_id,age,country,membership_status,previous_purchases,destination,stay_length,guests,travel_month,months_before_travel,earlybird_discount,package_Type,rating,margin,additional_services_cost,country_match,continent_match
39087,39088,24,Germany,silver,2,Tokyo,4,2,12,5,True,Adventure,5,1404.79,103.43,False,False
30893,30894,34,USA,standard,2,Paris,4,2,8,1,False,Cultural,5,1135.69,345.63,False,False
45278,45279,32,Italy,standard,2,Rio,4,3,2,4,True,Relaxation,5,199.89,86.47,False,False
16398,16399,53,Germany,silver,6,Rio,2,2,1,1,False,Adventure,5,64.28,-55.23,False,False
13653,13654,36,Germany,silver,3,Cairo,2,1,2,4,True,Relaxation,5,352.40,-75.80,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,11285,30,Italy,gold,6,Rome,2,1,6,3,False,Cultural,7,94.40,21.36,True,True
44732,44733,29,Germany,standard,1,Cape Town,2,2,2,3,False,Adventure,7,-157.90,109.68,False,False
38158,38159,33,Germany,standard,2,Cairo,6,2,2,3,False,Adventure,6,665.88,166.64,False,True
860,861,43,Germany,standard,4,Cape Town,1,2,2,2,False,Cultural,5,-271.69,136.77,False,False


We can later add this as a new preprocessing step to our column transformer.

💡 We briefly make a transformer called `do_nothing_to`. The semantics aren't important as this is something you will rarely need in practice and definitely not on the exam. All it does is make an anonymous function (a function without a name) that takes an input and returns that as output so it effectively does nothing. We make it a transformer by giving it to `FunctionTransformer`.

💡 The reason why this is necessary is that our columns transformer is configured to drop all unused columns. We need to "use" it for a column to stay in our dataset.

In [21]:
from sklearn.pipeline import FunctionTransformer


do_nothing_to = FunctionTransformer(lambda x: x)

In [22]:
preprocessing_country = make_column_transformer(
    (StandardScaler(), numeric_columns),
    (OneHotEncoder(sparse_output=False), cat_columns),
    (do_nothing_to,  ["country_match", "continent_match"]),
    remainder="drop"
)

##### Interaction terms

<img src="images/interaction_term.webp">

Let's consider an alternative approach to the problem above using interaction terms. Interaction terms are created by multiplying two variables together, effectively creating a new variable. This is a mathematical way of capturing the 'AND' condition in our data. For example, after one-hot encoding categorical variables like 'Country' and 'Destination', we can generate interaction terms to explore the combined effect of these two features.

Suppose we have 'New York', 'Rome', 'Tokyo', and 'Cairo' as categories for 'City' and 'USA', 'Italy', 'Japan', and 'Egypt' for 'Country'. If we create interaction terms for 'City' and 'Country', we end up with additional columns such as 'New York x USA', 'Rome x Italy', and so on. Each of these new columns will have a value of 1 only if both contributing variables (e.g., 'City' is 'New York' AND 'Country' is 'USA') are 1; otherwise, the value will be 0. This new variable thus answers the question: "Is the traveler from X city AND going to Y country?"

By including interaction terms, we allow our model to consider the combined influence of two variables, which can be particularly insightful when the effect of one variable on the outcome depends on the level of another variable.

In scikit-learn, interaction effects between variables can be encoded using the `PolynomialFeatures` transformer. Interaction effects are valuable in a model when the relationship between two features can affect the outcome in a way that is not simply additive.

For example, consider two binary features, A and B. Individually, they might have a certain effect on the target variable Y. However, when both A and B occur together (i.e., A=1 and B=1), their combined effect on Y could be different from the sum of their individual effects. This is where interaction terms come into play.

The PolynomialFeatures transformer can not only generate polynomial features, which are features raised to a power (like $x^{2}$ or $x^{3}$), but also interaction features, which are products of features (like $x_{1} * x_{2}$).

 Note: polynomial is called "veelterm" in Dutch.

We will start off by generating interactions between all our numeric columns.

##### ❓ How can you introduce polynomial features using scikit-learn? Try to setup a pipeline to include the country_match and continent_match we created above.

In [23]:
from sklearn.preprocessing import PolynomialFeatures

# Eerst schalen, dan alle paren van numerieke kolommen met elkaar vermenigvuldigen.
# interaction_only=True: enkel x1*x2, geen x1^2. include_bias=False: geen kolom met enkel 1-en.
numeric_interactions = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
)

preprocessing_interactions = make_column_transformer(
    (numeric_interactions, numeric_columns),
    (OneHotEncoder(sparse_output=False), cat_columns),
    (do_nothing_to, ["country_match", "continent_match"]),
    remainder="drop"
)

interactions_pipe = make_pipeline(country_map, preprocessing_interactions, LinearRegression())
interactions_pipe.fit(X_train, y_train)
predictions_interactions_test = interactions_pipe.predict(X_test)

print(f"MAE  test: {mean_absolute_error(y_test, predictions_interactions_test):.2f}")
print(f"RMSE test: {root_mean_squared_error(y_test, predictions_interactions_test):.2f}")

MAE  test: 282.64
RMSE test: 378.79


##### Binning

<img src="https://miro.medium.com/max/2000/1*LGTAObYYj2-fdBMFLz30rw.jpeg" style="width:50%">

Binning is a technique that involves segmenting a continuous variable into several intervals, or 'bins'. Similar to the way we categorize data when creating histograms, binning transforms a continuous variable into an ordered categorical variable. One hot encoding these bins allows us to introduce non-linear effects into our linear models, which ordinarily would interpret the data as having a constant slope.

Take temperature as an example: people generally enjoy mild increases in weather warmth, but there's a threshold beyond which higher temperatures become unpleasant. Binning would let us model this non-linear relationship. Instead of treating temperature as a single continuous predictor with a constant effect, we could divide temperatures into ranges (e.g., 0-10°C, 10-20°C, 20-30°C, etc.) and treat each range as a separate category. By one hot encoding these categories, we enable our model to capture the varying effects of different temperature ranges on people's comfort levels. This approach can reveal more complex patterns in how the predictor variable (in this case, temperature) influences the outcome variable (such as people's reported happiness).

##### ❓ How can you create bins for the variable `age` using scikit-learn? Add this preprocessing step to the column transformer.

In [24]:
# import the necessary tool
from sklearn.preprocessing import KBinsDiscretizer

# leeftijd opdelen in 6 even brede groepen en die one-hot encoden
age_bins = KBinsDiscretizer(n_bins=6, encode="onehot-dense", strategy="uniform")
numeric_without_age = ["guests", "stay_length"]

preprocessing_bins = make_column_transformer(
    (StandardScaler(), numeric_without_age),
    (age_bins, ["age"]),
    (OneHotEncoder(sparse_output=False), cat_columns),
    (do_nothing_to, ["country_match", "continent_match"]),
    remainder="drop"
)

# make a preprecossing_bins_interactions column transformer to bring everything together
preprocessing_bins_interactions = make_column_transformer(
    (numeric_interactions, numeric_columns),
    (age_bins, ["age"]),
    (OneHotEncoder(sparse_output=False), cat_columns),
    (do_nothing_to, ["country_match", "continent_match"]),
    remainder="drop"
)

# controleren welke grenzen de bins hebben
age_bins.fit(X_train[["age"]]).bin_edges_

array([array([18.        , 28.33333333, 38.66666667, 49.        , 59.33333333,
              69.66666667, 80.        ])                                      ],
      dtype=object)

#### Finding the best model: cross validation

As we explore various models and preprocessing techniques, we encounter a dilemma: how do we identify the best model without biasing our selection? 

##### ❓ What is the weakness of the train-test split approach? Think about this before continuing.

We kijken maar naar één willekeurige opsplitsing. Als we toevallig een "makkelijke" of "moeilijke" testset hebben, is het resultaat te optimistisch of te pessimistisch.

Daarnaast: als we veel modellen en preprocessing-varianten uitproberen en telkens het beste kiezen op basis van de testset, dan gebruiken we de testset eigenlijk om ons model te kiezen. Dan is de testset niet meer "ongezien" en is onze eindscore te optimistisch (een soort data leakage).

##### ❓ Think about a solution for this before we continue.

De testset opzij leggen tot helemaal op het einde, en de keuze tussen modellen maken op basis van een deel van de *trainingsset* (een validatieset). Nog beter: dat meerdere keren doen met telkens een ander stuk als validatieset en het gemiddelde nemen, zodat het resultaat niet afhangt van één toevallige opsplitsing.

The solution we will use is called cross-validation.
Here's how it works: We divide our training set into smaller sections, say 20% chunks. We train our model on 80% of the data, then validate it on the remaining 20%. We repeat this process five times, each time with a different 20% held out for validation. This technique, known as k-fold cross-validation (with k being the number of chunks or 'folds' we create), allows each model a fair shot at proving itself across the entirety of our data.

By averaging the performance across these folds, we obtain a more reliable measure of a model's quality. This thorough approach increases our confidence that we're selecting the best model, not by chance, but by consistent performance.

<center><img src="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" style="background-color:white"></center>

##### ❓ Sanity check: if we do K-fold cross validation, how many models have we trained. Answer for 2-fold, 3-fold, 5-fold and K-fold.

Bij K-fold cross validation trainen we K modellen (elke fold is één keer de validatieset):
* 2-fold → 2 modellen
* 3-fold → 3 modellen
* 5-fold → 5 modellen
* K-fold → K modellen

(+ eventueel nog één keer op de volledige trainingsset voor het finale model.)

##### ❓ What is the downside of K-fold cross validation?

Het kost veel meer rekentijd: je traint K keer in plaats van één keer. Voor trage modellen (bv. random forest op een grote dataset) en zeker in combinatie met hyperparameter tuning loopt dat snel op.

As usual, sci-kit learn makes it easy to do cross validation. 

We supply `cross_val_score` with 3 mandatory parameters:

* The machine learning model
* The `X_train`
* `X_test`
  
Additionally you can use `cv` to specify how many folds and you can pick a `score` parameter, which is the result that will be reported to you as the performance on each fold.

##### ❓ How do you include cross validation using scikit-learn?

In [25]:
# import the appropriate tool
from sklearn.model_selection import cross_val_score

# calculate the cross validation score for the linear regression pipeline
# you'll need to choose a "scoring" metric
# sklearn wil altijd "hoger = beter", daarom is de RMSE hier negatief
lin_reg_scores = cross_val_score(lin_reg_pipe, X_train, y_train, cv=5,
                                 scoring="neg_root_mean_squared_error", n_jobs=-1)
lin_reg_scores

array([-407.45868715, -430.29052685, -415.38436966, -415.98848015,
       -408.92248265])

In [26]:
# calculate the average result from the scores above
print(f"Gemiddelde RMSE linear regression: {-np.mean(lin_reg_scores):.2f} (std {np.std(lin_reg_scores):.2f})")

Gemiddelde RMSE linear regression: 415.61 (std 8.09)


In [27]:
# build a decision tree pipeline
tree_pipe = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))

In [28]:
# repeat the cross validation scoring and average result, this time for decision trees
tree_scores = cross_val_score(tree_pipe, X_train, y_train, cv=5,
                              scoring="neg_root_mean_squared_error", n_jobs=-1)
print(tree_scores)
print(f"Gemiddelde RMSE decision tree: {-np.mean(tree_scores):.2f} (std {np.std(tree_scores):.2f})")

[-422.80409362 -430.55284122 -415.95484427 -422.78689026 -408.23643751]
Gemiddelde RMSE decision tree: 420.07 (std 7.51)


##### ❓ Interpret these values. Which model performs better?

Lineaire regressie heeft een gemiddelde RMSE van ±416, de decision tree ±420. Ze zijn dus ongeveer even goed; lineaire regressie is heel licht beter. De standaardafwijking over de folds is klein (±8), dus de resultaten zijn stabiel over de verschillende folds.

##### ❓ Contrast the performance seen in the beginning of this notebook of these two models. Think about overfitting and so on. (Key Insight!)

In het begin van het notebook leek de decision tree op de **trainingsset** veel beter (MAE ±12 tegenover ±307). Met cross validation zien we dat dat volledig misleidend was: op data die de boom niet gezien heeft, is hij niet beter (eerder slechter) dan lineaire regressie. De boom overfit dus.

**Key insight:** de score op de trainingsset zegt niets over hoe goed een model generaliseert. Cross validation geeft ons een eerlijke schatting op ongeziene data, zonder dat we de testset moeten aanspreken.

##### ❓ Your turn: experiment with different models and setups. Try out the new pre processing pipelines with a variety of machine learning models and cross validation. Feel free to reuse some of the evaluation code of the beginning of the lecture. 

In [29]:
preprocessing_country
preprocessing_interactions
preprocessing_bins
preprocessing_bins_interactions;

# Your pipeline should look like this: pipe = make_pipeline(country_map, prep, model)
# Deviations from this will likely cause an error.


def cross_validate_models(models: dict, preprocessors: dict, X, y, scoring: str, cv: int = 5) -> pd.DataFrame:
    """Doet cross validation voor elke combinatie van preprocessing en model.
    Geeft per combinatie het gemiddelde en de standaardafwijking van de scores terug."""
    rows = []
    for prep_name, prep in preprocessors.items():
        for model_name, model in models.items():
            pipe = make_pipeline(country_map, prep, model)
            scores = cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
            rows.append({"preprocessing": prep_name, "model": model_name,
                         "mean": np.mean(scores), "std": np.std(scores)})
    return pd.DataFrame(rows)


preprocessors = {
    "basis": preprocessing,
    "country": preprocessing_country,
    "interactions": preprocessing_interactions,
    "bins": preprocessing_bins,
    "bins + interactions": preprocessing_bins_interactions,
}

cv_results = cross_validate_models(models, preprocessors, X_train, y_train,
                                   scoring="neg_root_mean_squared_error")
cv_results["RMSE"] = -cv_results["mean"]  # terug positief maken zodat het leesbaar is
cv_results.pivot(index="preprocessing", columns="model", values="RMSE").round(1)

model,Decision Tree,Gradient Boosting,Linear Regression,Random Forest
preprocessing,,,,
basis,420.10,285.40,415.60,316.80
bins,419.70,301.20,388.80,335.40
bins + interactions,412.80,284.10,382.90,309.50
country,411.60,283.60,391.20,310.30
interactions,411.90,284.50,386.10,309.70


Het beste model uit de cross validation trainen we nu op de volledige trainingsset en evalueren we één keer op de testset.

In [30]:
best_row = cv_results.sort_values("RMSE").iloc[0]
print(f"Beste combinatie: {best_row['model']} met preprocessing '{best_row['preprocessing']}'")

best_pipe = make_pipeline(country_map, preprocessors[best_row["preprocessing"]], models[best_row["model"]])
best_pipe.fit(X_train, y_train)
best_predictions = best_pipe.predict(X_test)

print(f"MAE  test: {mean_absolute_error(y_test, best_predictions):.2f}")
print(f"RMSE test: {root_mean_squared_error(y_test, best_predictions):.2f}")

Beste combinatie: Gradient Boosting met preprocessing 'country'


MAE  test: 221.15
RMSE test: 280.19


##### ❓ Is our new approach of using interaction terms and binning an improvement over before?

Deels wel:
* **Lineaire regressie** wint het meest: de RMSE gaat van ±416 (basis) naar ±383 met bins + interactions. Een lineair model kan uit zichzelf geen niet-lineaire verbanden of "EN"-combinaties leren, dus extra features helpen hier echt.
* **Gradient boosting** en **random forest** verbeteren nauwelijks (±285 → ±284, ±317 → ±310). Boom-modellen kunnen zelf al splitsingen op leeftijd maken en combinaties van features leren, dus binning en interactions voegen weinig toe. Alleen bins (zonder interactions) is voor die modellen zelfs slechter: door de leeftijd in 6 groepen te steken gooi je informatie weg.
* De eenvoudige `country_match` / `continent_match` features helpen bij alle modellen een beetje.

Beste combinatie uit de cross validation: gradient boosting met `preprocessing_country`. Op de testset geeft dat een RMSE van ±280 en een MAE van ±221, wat in lijn ligt met de cross validation score. Er is dus geen teken dat we te optimistisch waren.

#### Finding the best model: can we do more?

We have already tried different methods like creating new features and grouping data. These methods have sometimes made our models better. But, there's more we can do.

Machine learning is a process that goes in circles, as shown in the CRISP-DM image. Once we build models, there are ways to look more closely at how they work.

1. Comparing Predicted and Actual Results
   
The first step we did was to compare what our model predicted with the real results. This helps us see where the model does well and where it needs more work. It also shows anomalies, for instance certain models were predicting negative costs which should not be possible.

2. Looking at Residuals
   
Here, we look at the errors - the difference between the real values (y_true) and what the model predicted (y_pred). We do this for each input in our model. The main idea is to look for patterns in these errors. For example, if we make charts showing errors for each country and notice that errors are bigger for certain countries, it tells us that we might need to improve our features. This is like doing analysis on the results of our model.

##### Parameters and hyperparameters (only theory)

Another thing we can do is called **hyperparameter tuning**. Hyperparameters are simply the "settings" of the machine learning model. Parameters are what it uses to make predictions. So for linear regression the parameters are the coefficients and for decision trees these are the splits it is making. **Parameters are what the model learns from the data, while hyperparameters are what we decide before training starts.**

Most machine learning models have hyperparameters (settings) as well. Hyperparameters control how flexible or constrained a model is. Typically we adjust hyperparameters to decrease the complexity of the model (make it simpler) to prevent overfitting, but they can also be adjusted to make a model more complex to capture patterns better. For a decision tree a hyperparameter is for instance the amount of splits it is allowed to make. For Random Forest and Gradient boosting the amount of trees it uses are a hyperparameter. For linear regression the regularization constant (punishment if the coefficients get large) is also a hyperparameter. **We adjust hyperparameters to find a good balance between underfitting and overfitting.**


Typically complex models have many different hyperparameters you can tune. Tuning hyperparameters is like adjusting the knobs on a radio — you’re trying to find the setting that gives the clearest signal (best performance) without noise (overfitting). We encourage you to look at the number of hyperparameters for the models we have used so far to get an idea:

* [Linear regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
* [Gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)
* [Random forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
* [Decision tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)


Some models are very sensitive to their choice of hyperparameters. With their defaults they typically overfit, this is the case for random forest and decision trees (see above).

To identify the best hyperparameters, techniques like grid search and random search are used. Grid search exhaustively explores all possible combinations of hyperparameters, offering a thorough but computationally intensive approach. Random search, on the other hand, samples a predetermined number of combinations randomly, providing a quicker but slightly less focused alternative.

<center>
<img src="https://www.researchgate.net/publication/341691661/figure/fig2/AS:896464364507139@1590745168758/Comparison-between-a-grid-search-and-b-random-search-for-hyper-parameter-tuning-The.png"></center>


##### ❓ hyperparameters can be sensitive to specific parts of the data. We don't want hyperparameters that do well on one specific part of the data. What can we do to make sure we select hyperparameters that work well for the entire dataset?

Hyperparameters kiezen we ook met **cross validation**: voor elke combinatie van hyperparameters doen we K-fold cross validation op de trainingsset en nemen we de gemiddelde score. Zo kiezen we instellingen die over alle folds goed werken in plaats van op één toevallige split (in sklearn bv. met `GridSearchCV` of `RandomizedSearchCV`). De testset blijft daarbij helemaal buiten schot.

**The key question is if manually doing grid search or random search is worth it.** The answer, as usual, is it depends.

Counter-arguments: 

1. Linear regression in sci-kit learn automatically performs hyperparameter tuning:

* [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)
* [ElasticNetCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNetCV.html)
   
Their names come from the specific regularization they perform, covering that is out of scope for this course.

The CV stands for "cross validation". These models perform hyperparameter tuning with cross validation by default. For linear regression this is fine as this is a very inexpensive model to train.

2. It is very time consuming.
3. The difference can be negligible. You also don't know ahead of time if it will matter.
4. It requires you to somewhat know the internal details of the models.
5. Gradient boosting is extremely powerful with the default settings, it may not even require tuning.


Arguments for:

1. In some cases a 1 % difference absolutely makes a difference.

This is related to point 3. Even if the difference is negligible at a certain scale a 1 % increase matters. For instance, if you have a ML model to predict how much food will be thrown out for a small grocery store a difference of 1 % does not really matter. If this is on the scale of the entire franchise then it is a worthwhile investment.

### Summary

Revisiting the complex image we began with, after our lab today, the process should be clearer:

<center>
<img src="https://scikit-learn.org/stable/_images/grid_search_workflow.png" style="background-color:white;width:50%">
</center>


1. You start by dividing your dataset into two parts: training and test data.
2. The training data is then used for cross-validation. This technique is a reliable way to assess model performance and can be paired with hyperparameter tuning, although that's not mandatory.
3. Choose the model or models that perform best in the cross-validation phase.
4. These top models are then trained with the entire set of training data.
5. finally, we test these models on the test data. The results from this step give us our final evaluation metrics.


Remember, this method, aside from the optional step of hyperparameter tuning, is crucial for your exam. Evaluating and training on the same data is a practice we want to avoid because it can lead to overfitting, which is why we test our models on unseen data.